In [1]:
import random
import time
from typing import Callable, Literal, Optional, TypedDict
from langgraph.graph import END, START, StateGraph

# ==========================================
# 1. 扩展后的 AgentState (包含 Bulkhead 标记)
# ==========================================
class AgentState(TypedDict):
    employee_id: str
    amount: float

    # API Attempt 状态
    api_status: Optional[int]
    error_msg: Optional[str]

    # Half-Open & Bulkhead 状态
    is_half_open_probe: bool
    holding_bulkhead_slot: bool  # 是否持有并发槽位

    # Retry 策略状态
    retry_count: int
    max_retries: int
    retry_delay: float

    # Deadline 策略状态
    deadline: float
    request_timeout: float

    # External Interruption 状态
    cancelled: bool

    # Policy 决策输出
    policy_action: Optional[
        Literal[
            "ALLOW",
            "FAST_FAIL",
            "BULKHEAD_REJECTED",
            "SUCCESS",
            "RETRY",
            "FALLBACK",
            "DEADLINE_EXCEEDED",
            "CANCELLED",
            "FATAL_ERROR",
        ]
    ]

    # 执行结果
    result: Optional[str]


# ==========================================
# 2. 共享 Bulkhead 模式实现
# ==========================================
class Bulkhead:
    def __init__(self, capacity: int):
        self.capacity: int = capacity
        self.in_flight: int = 0

    def try_acquire(self) -> bool:
        if self.in_flight < self.capacity:
            self.in_flight += 1
            return True
        return False

    def release(self) -> None:
        if self.in_flight > 0:
            self.in_flight -= 1


class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown: float = 5.0):
        self.state: str = "CLOSED"
        self.failure_count: int = 0
        self.failure_threshold: int = failure_threshold
        self.cooldown: float = cooldown
        self.opened_at: Optional[float] = None

    def can_call(self) -> bool:
        now = time.time()
        if self.state == "OPEN":
            if self.opened_at and (now - self.opened_at >= self.cooldown):
                self.state = "HALF_OPEN"
                return True
            return False
        return True

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        if self.state == "HALF_OPEN" or self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.opened_at = time.time()


# 全局 Runtime 共享实例
hr_api_breaker = CircuitBreaker()
hr_api_bulkhead = Bulkhead(capacity=2)


# ==========================================
# 3. 核心 Node 实现
# ==========================================
def circuit_breaker_gate_node(state: AgentState) -> dict:
    allowed = hr_api_breaker.can_call()
    if not allowed:
        return {"policy_action": "FAST_FAIL", "is_half_open_probe": False}

    is_probe = hr_api_breaker.state == "HALF_OPEN"
    return {"policy_action": "ALLOW", "is_half_open_probe": is_probe}


def bulkhead_acquire_node(state: AgentState) -> dict:
    """尝试获取 Bulkhead 并发槽位"""
    acquired = hr_api_bulkhead.try_acquire()
    if acquired:
        return {"holding_bulkhead_slot": True}
    return {"policy_action": "BULKHEAD_REJECTED", "holding_bulkhead_slot": False}


def call_hr_api_node(state: AgentState) -> dict:
    """单次 API 调用 (包含异常处理保护)"""
    try:
        status = state.get("api_status", 503)
        if status == 999:  # 模拟代码内部抛出未捕获异常
            raise RuntimeError("Network Interface Crashed!")
        error_msg = "OK" if status == 200 else f"HTTP Error {status}"
        return {"api_status": status, "error_msg": error_msg}
    except Exception as e:
        return {"api_status": 500, "error_msg": str(e)}


def bulkhead_release_node(state: AgentState) -> dict:
    """安全释放 Bulkhead 槽位：只要成功持有过，就必须释放"""
    if state.get("holding_bulkhead_slot", False):
        hr_api_bulkhead.release()
        return {"holding_bulkhead_slot": False}
    return {}


def policy_node(state: AgentState) -> dict:
    """策略决策中心"""
    # 1. 外部取消判定
    if state.get("cancelled", False):
        return {"policy_action": "CANCELLED"}

    # Bulkhead 限流被拒判定（注意：Bulkhead 被拒不影响 Circuit Breaker 状态）
    if state.get("policy_action") == "BULKHEAD_REJECTED":
        return {"policy_action": "FALLBACK", "result": "Bulkhead Capacity Full -> Fallback"}

    status = state.get("api_status")

    # 2. 成功路径
    if status == 200:
        hr_api_breaker.record_success()
        return {"policy_action": "SUCCESS", "result": "API Call Succeeded"}

    # 失败反馈给 Circuit Breaker
    hr_api_breaker.record_failure()

    # 3. 不可重试致命错误
    if status in (400, 401, 403, 404, 422, 501):
        return {"policy_action": "FATAL_ERROR", "result": f"Fatal Error: {status}"}

    # HALF_OPEN Probe 失败立刻 Fallback
    if state.get("is_half_open_probe", False):
        return {"policy_action": "FALLBACK", "result": "Half-Open Probe Failed -> Fallback"}

    # 4. 重试上限检查
    if state["retry_count"] >= state["max_retries"]:
        return {"policy_action": "FALLBACK", "result": "Max Retries Reached -> Fallback"}

    # 5. Backoff + Jitter 计算
    backoff = state.get("retry_delay", 0.1) * (2 ** state["retry_count"])
    jitter = random.uniform(0.0, 0.05)
    computed_delay = backoff + jitter

    # 6. Deadline 预算检查
    now = time.time()
    remaining_budget = state["deadline"] - now
    required_budget = computed_delay + state["request_timeout"] + 0.1

    if remaining_budget < required_budget:
        return {"policy_action": "DEADLINE_EXCEEDED", "result": "Deadline Exceeded -> Fallback"}

    return {"policy_action": "RETRY", "retry_delay": computed_delay}


def retry_wait_node(state: AgentState) -> dict:
    time.sleep(state["retry_delay"])
    return {"retry_count": state["retry_count"] + 1}


def fallback_node(state: AgentState) -> dict:
    return {"result": f"Fallback Executed. Action: {state.get('policy_action')}"}


# ==========================================
# 4. Router 与 Graph 编排
# ==========================================
def route_gate(state: AgentState) -> str:
    if state["policy_action"] == "ALLOW":
        return "bulkhead_acquire"
    return "fallback"


def route_acquire(state: AgentState) -> str:
    if state["holding_bulkhead_slot"]:
        return "call_hr_api"
    return "policy"


def route_policy(state: AgentState) -> str:
    action = state["policy_action"]
    if action == "SUCCESS":
        return "end"
    if action == "RETRY":
        return "retry_wait"
    if action in ("FALLBACK", "DEADLINE_EXCEEDED"):
        return "fallback"
    if action in ("CANCELLED", "FATAL_ERROR"):
        return "end"
    return "end"


builder = StateGraph(AgentState)

builder.add_node("circuit_breaker_gate", circuit_breaker_gate_node)
builder.add_node("bulkhead_acquire", bulkhead_acquire_node)
builder.add_node("call_hr_api", call_hr_api_node)
builder.add_node("bulkhead_release", bulkhead_release_node)
builder.add_node("policy", policy_node)
builder.add_node("retry_wait", retry_wait_node)
builder.add_node("fallback", fallback_node)

builder.set_entry_point("circuit_breaker_gate")

# Breaker 路由
builder.add_conditional_edges(
    "circuit_breaker_gate",
    route_gate,
    {"bulkhead_acquire": "bulkhead_acquire", "fallback": "fallback"},
)

# Bulkhead Acquire 路由
builder.add_conditional_edges(
    "bulkhead_acquire",
    route_acquire,
    {"call_hr_api": "call_hr_api", "policy": "policy"},
)

# 无论 API 执行成功与否/发生异常，必定经过 Release 节点
builder.add_edge("call_hr_api", "bulkhead_release")
builder.add_edge("bulkhead_release", "policy")

# Policy 路由
builder.add_conditional_edges(
    "policy",
    route_policy,
    {
        "end": END,
        "retry_wait": "retry_wait",
        "fallback": "fallback",
    },
)

# Retry 回环：重新获取 Bulkhead 槽位
builder.add_edge("retry_wait", "bulkhead_acquire")
builder.add_edge("fallback", END)

graph = builder.compile()


# ==========================================
# 5. 4 个核心 Scenario 验证
# ==========================================
if __name__ == "__main__":
    def create_state(**kwargs) -> AgentState:
        default: AgentState = {
            "employee_id": "EMP_001",
            "amount": 100.0,
            "api_status": 200,
            "error_msg": None,
            "is_half_open_probe": False,
            "holding_bulkhead_slot": False,
            "retry_count": 0,
            "max_retries": 1,
            "retry_delay": 0.01,
            "deadline": time.time() + 10.0,
            "request_timeout": 1.0,
            "cancelled": False,
            "policy_action": None,
            "result": None,
        }
        default.update(kwargs)
        return default

    print("=== Scenario 1: 有槽位 → 正常调用 → release ===")
    hr_api_bulkhead = Bulkhead(capacity=2)
    res1 = graph.invoke(create_state(api_status=200))
    print(f"Result: {res1.get('result')}")
    print(f"In-Flight after call: {hr_api_bulkhead.in_flight} (Expected: 0)\n")

    print("=== Scenario 2: Bulkhead 满 → 不调用 HR API → 不影响 Circuit Breaker ===")
    hr_api_bulkhead = Bulkhead(capacity=1)
    hr_api_breaker = CircuitBreaker()
    
    # 占满槽位
    hr_api_bulkhead.try_acquire()
    print(f"Before call, in_flight: {hr_api_bulkhead.in_flight}")

    res2 = graph.invoke(create_state(api_status=200))
    print(f"Action: {res2.get('policy_action')} | Result: {res2.get('result')}")
    print(f"Breaker State: {hr_api_breaker.state} (Expected: CLOSED, 不受影响)\n")

    print("=== Scenario 3: API 失败/异常 → 即使失败也 release slot ===")
    hr_api_bulkhead = Bulkhead(capacity=2)
    
    # 模拟 API 抛出未捕获异常 (999)
    res3 = graph.invoke(create_state(api_status=999, max_retries=0))
    print(f"Action: {res3.get('policy_action')} | Result: {res3.get('result')}")
    print(f"In-Flight after error: {hr_api_bulkhead.in_flight} (Expected: 0, 无 Leak)\n")

    print("=== Scenario 4: 多次调用/包含重试后 in_flight 最终回到 0 ===")
    hr_api_bulkhead = Bulkhead(capacity=2)
    hr_api_breaker = CircuitBreaker()

    # 经历一次 503 重试后成功
    res4 = graph.invoke(create_state(api_status=503, max_retries=2))
    print(f"Retries: {res4.get('retry_count')} | Action: {res4.get('policy_action')}")
    print(f"Final In-Flight: {hr_api_bulkhead.in_flight} (Expected: 0)\n")

=== Scenario 1: 有槽位 → 正常调用 → release ===
Result: API Call Succeeded
In-Flight after call: 0 (Expected: 0)

=== Scenario 2: Bulkhead 满 → 不调用 HR API → 不影响 Circuit Breaker ===
Before call, in_flight: 1
Action: FALLBACK | Result: Fallback Executed. Action: FALLBACK
Breaker State: CLOSED (Expected: CLOSED, 不受影响)

=== Scenario 3: API 失败/异常 → 即使失败也 release slot ===
Action: FALLBACK | Result: Fallback Executed. Action: FALLBACK
In-Flight after error: 0 (Expected: 0, 无 Leak)

=== Scenario 4: 多次调用/包含重试后 in_flight 最终回到 0 ===
Retries: 2 | Action: FALLBACK
Final In-Flight: 0 (Expected: 0)

